# Geometric Brownian Motion Demo

This notebook demonstrates the key features of the `gbm` package:
- Parameter calibration from historical stock data
- GBM path simulation
- Option pricing (Black-Scholes and Monte Carlo)
- Greeks calculation (Delta, Gamma, Vega, Theta)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gbm import GeometricBrownianMotion, black_scholes, monte_carlo_price, calibrate_from_ticker, delta, gamma, vega, theta


## Parameter Calibration

Calibrate drift (μ) and volatility (σ) from historical stock data using Yahoo Finance.

In [ ]:
# Calibrate parameters for Apple stock
params = calibrate_from_ticker('AAPL', period='2y')
print("Calibrated parameters:")
for key, value in params.items():
    print(f"{key}: {value}")

# Create GBM instance
gbm = GeometricBrownianMotion(S0=params['S0'], mu=params['mu'], sigma=params['sigma'])

## GBM Path Simulation

Simulate multiple paths of the stock price over time.

In [ ]:
# Simulate 100 paths over 1 year
t, paths = gbm.simulate(T=1.0, dt=1/252, n_paths=100, seed=42)

plt.figure(figsize=(12, 6))
plt.plot(t, paths.T, alpha=0.6, linewidth=0.8)
plt.plot(t, gbm.S0 * np.exp(gbm.mu * t), 'k--', linewidth=2, label='Expected path')
plt.title(f'GBM Simulation for AAPL\nμ={gbm.mu:.4f}, σ={gbm.sigma:.4f}, S₀={gbm.S0:.2f}')
plt.xlabel('Time (years)')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Option Pricing

Compare Black-Scholes closed-form pricing with Monte Carlo simulation.

In [ ]:
# Option parameters
S = gbm.S0
K = S * 1.05  # 5% out of the money
T = 0.5  # 6 months
r = 0.05  # risk-free rate
sigma = gbm.sigma

# Black-Scholes price
bs_call = black_scholes(S, K, T, r, sigma, 'call')
bs_put = black_scholes(S, K, T, r, sigma, 'put')

# Monte Carlo price
mc_call, mc_call_std = monte_carlo_price(S, K, T, r, sigma, n_paths=50000, option_type='call')
mc_put, mc_put_std = monte_carlo_price(S, K, T, r, sigma, n_paths=50000, option_type='put')

print(f"Strike Price: ${K:.2f}")
print(f"Time to Expiry: {T*12:.0f} months")
print()
print("Black-Scholes Prices:")
print(f"Call: ${bs_call:.4f}")
print(f"Put:  ${bs_put:.4f}")
print()
print("Monte Carlo Prices (50k paths):")
print(f"Call: ${mc_call:.4f} ± ${mc_call_std:.4f}")
print(f"Put:  ${mc_put:.4f} ± ${mc_put_std:.4f}")

## Greeks Calculation

Calculate option sensitivities: Delta, Gamma, Vega, Theta.

In [ ]:
# Calculate Greeks for call option
call_delta = delta(S, K, T, r, sigma, 'call')
call_gamma = gamma(S, K, T, r, sigma)
call_vega = vega(S, K, T, r, sigma)
call_theta = theta(S, K, T, r, sigma, 'call')

# For put option
put_delta = delta(S, K, T, r, sigma, 'put')
put_gamma = gamma(S, K, T, r, sigma)
put_vega = vega(S, K, T, r, sigma)
put_theta = theta(S, K, T, r, sigma, 'put')

print("Call Option Greeks:")
print(f"Delta: {call_delta:.4f}")
print(f"Gamma: {call_gamma:.4f}")
print(f"Vega:  {call_vega:.4f}")
print(f"Theta: {call_theta:.4f}")
print()
print("Put Option Greeks:")
print(f"Delta: {put_delta:.4f}")
print(f"Gamma: {put_gamma:.4f}")
print(f"Vega:  {put_vega:.4f}")
print(f"Theta: {put_theta:.4f}")